### Imports

In [367]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

### Basic Intuition

In [ ]:
# read the data
names = open("../names.txt", 'r').read().splitlines()
names[:8]

In [ ]:
len(names)

In [ ]:
# create the vocabulary
chars = sorted(list(set(''.join(names))))
stoi = {s:i for i, s in enumerate(chars, start=1)}
stoi["."] = 0
itos = {i:s for s,i in stoi.items()}
itos

In [ ]:
# build dataset
block_size = 3
X, Y = [],[]

for w in names[:5]:
    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '----->',itos[ix])
        context = context[1:] + [ix]

X, Y = torch.tensor(X), torch.tensor(Y)

In [ ]:
# (examples, block_size)
X[:5],Y[:5]

In [373]:
# lookup table C
g = torch.Generator().manual_seed(42)
C = torch.randn(size=(27, 2), generator=g)

In [374]:
dummy_encoding = F.one_hot(torch.tensor(5), num_classes=27).float()
assert (C[5] == (dummy_encoding @ C)).all()

In [ ]:
# encode the context again
X_enc = F.one_hot(X, num_classes=27).float()
X_enc.shape

In [ ]:
# get lookup values for multiple inputs
C[[5,6,7]]

In [ ]:
C[X].shape

In [ ]:
# get embedding (examples, block size, embedding size)
emb = C[X]
emb.shape

In [379]:
# build the model
g = torch.Generator().manual_seed(42)
W1 = torch.randn(size=(6, 100), generator=g)
b1 = torch.randn(100, generator=g)


In [ ]:
# get the embeddings for all the 32 examples and concatenate them so that we can get the shape of (examples, block_size * embedding_size)
# emb shape -> (examples, block_size, embedding_size)
torch.cat(torch.unbind(emb, dim=1), dim=1).shape

In [34]:
# check if element wise both the operations are equviavalent.
assert (torch.cat(torch.unbind(emb, 1), 1) == emb.view(32, -1)).all()

In [ ]:
# using the view method since its much more efficient
# in general -> we can view it as (examples, -1) the -1 makes it easy so we dont hard code things
h = torch.tanh(((emb.view(32,-1) @ W1) + b1))
h.shape

In [36]:
# output layer
g = torch.Generator().manual_seed(42)
W2 = torch.randn((100,27), generator=g)
b2 = torch.randn(27, generator=g)

In [ ]:
logits = ((h @ W2) + b2)
logits.shape

In [ ]:
probs = F.softmax(logits, dim = 1)

In [ ]:
F.cross_entropy(logits, Y)

### Putting it together

In [341]:
# HYPERPARAMS
BLOCK_SIZE = 3
EMBED_SIZE = 2
BOUNDARY_CHAR = "."
SEED_VAL = 42
BATCH_SIZE = 32

In [ ]:
# load the names
names = open("../names.txt").read().splitlines()
len(names), names[:8]

In [ ]:
# create the vocabulary
# get the unique characters in the names
unique_chars = sorted(list(set(''.join(names))))
unique_chars.insert(0, BOUNDARY_CHAR)
# create a mapping str -> int
stoi = {s: i for i , s in enumerate(unique_chars)}
# create the reverse mapping
itos = {i: s for s, i in stoi.items()}

len(unique_chars), stoi



In [ ]:
# create data set on the complete data set using the BLOCK_SIZE parameter
X, Y = [], []


for w in names:
    contex = [0] * BLOCK_SIZE
    for ch in w + BOUNDARY_CHAR:
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]

X, Y = torch.tensor(X), torch.tensor(Y)
X.shape, Y.shape

In [ ]:
# Create model architecture as defined below
# a lookup table that takes in the words (context) and embeds them into embed size vectors
# a neruon layer with 100 neurons that has the tanh actiavtion
# a output layer that has softmax at the end for probs


# generator object for reporoducible results
g = torch.Generator().manual_seed(SEED_VAL)


# lookup table we have len(unique_chars) unique characters (26 lower case alphabet + 1 boundary char)
C = torch.randn(size=(len(unique_chars), EMBED_SIZE), generator=g)


# the first layer contains W1 weights and b1 bias
# the shape of the W1 matrix will be (context_lenght or block_size * embed_dim, n_neurons)
# each neuron in the layer has a unique bias term so there will be n_neuron bias term
W1 = torch.randn(
    size = (
        BLOCK_SIZE * EMBED_SIZE,
        100
    ),
    generator=g
)

b1 = torch.randn(100)


# the output layer is a linear layer hence will contain its W matrix and b vector
# W2 will be of size -> (n_neurons_first_layer, n_unique) trying to model the probability for the next char given 27 characters and context
# b2 will be of size -> n_unique
W2 = torch.randn(
    size = (
        100,
        len(unique_chars)
    ),
    generator=g
)

b2 = torch.randn(
    len(unique_chars),
    generator=g
)


parameters = [C, W1,b1,W2,b2]

# add requires grad for all trainable parameters
for p in parameters:
    p.requires_grad = True

# C.shape -> (unique_chars, EMBED_SIZE)
# W1.shape -> (context_size * embed_dim, n_neurons_in_first_layer)
# b1.shape -> n_nuerons in first layer (each neuron has its own bias term)
C.shape, W1.shape, b1.shape, W2.shape, b2.shape

In [ ]:
# printing the numebr of parameters
sum(p.nelement() for p in parameters)

In [ ]:
# we index into the lookup table rater than one hot encode since its more easy
# also it allows to parallely embed the complete training set (or any set of input)

emb = C[X]

# shape -> (examples, context, embed)
# view shape -> efficient reshape of the tensors in emb so that we can do a dot product with the weights -> (examples, context_size * embed_size)
emb.shape, emb.view(len(X), -1).shape, W1.shape

In [ ]:
# get a good learning rate


# start with the low of 0.001 uptil 1.
lre = torch.linspace(-3, 0, 1000)
lrs = 10 ** lre

lossi = []
lri = []


# complete training loop

for i in range(1000):
    
    # get mini batch
    ix = torch.randint(0, X.shape[0], size=(BATCH_SIZE,))


    # forward pass
    # embed the input contexts
    emb = C[X[ix]]

    # run the embeddings through the first layer

    h = torch.tanh((emb.view(-1, BLOCK_SIZE * EMBED_SIZE) @ W1) + b1)

    # get the logits from the output layer
    logits = (h @ W2) + b2

    # calculate the loss
    loss = F.cross_entropy(logits, Y[ix])

    # backward pass

    # zero out the gradients
    for p in parameters:
        p.grad = None



    # backprop
    loss.backward()

    lr = lrs[i]
    # update
    for p in parameters:
        p.data = p.data - lr * p.grad   

    # track stats
    lossi.append(loss.item())
    lri.append(lre[i].item())

print(f"{loss=:.4f}")



In [ ]:
plt.plot(lri, lossi)

In [350]:
# the lr with min loss
LEARNING_RATE = 10 ** lri[torch.argmin(torch.tensor(lossi)).item()]

In [ ]:
# Create model architecture as defined below
# a lookup table that takes in the words (context) and embeds them into embed size vectors
# a neruon layer with 100 neurons that has the tanh actiavtion
# a output layer that has softmax at the end for probs


# generator object for reporoducible results
g = torch.Generator().manual_seed(SEED_VAL)


# lookup table we have len(unique_chars) unique characters (26 lower case alphabet + 1 boundary char)
C = torch.randn(size=(len(unique_chars), EMBED_SIZE), generator=g)


# the first layer contains W1 weights and b1 bias
# the shape of the W1 matrix will be (context_lenght or block_size * embed_dim, n_neurons)
# each neuron in the layer has a unique bias term so there will be n_neuron bias term
W1 = torch.randn(
    size = (
        BLOCK_SIZE * EMBED_SIZE,
        100
    ),
    generator=g
)

b1 = torch.randn(100)


# the output layer is a linear layer hence will contain its W matrix and b vector
# W2 will be of size -> (n_neurons_first_layer, n_unique) trying to model the probability for the next char given 27 characters and context
# b2 will be of size -> n_unique
W2 = torch.randn(
    size = (
        100,
        len(unique_chars)
    ),
    generator=g
)

b2 = torch.randn(
    len(unique_chars),
    generator=g
)


parameters = [C, W1,b1,W2,b2]

# add requires grad for all trainable parameters
for p in parameters:
    p.requires_grad = True

# C.shape -> (unique_chars, EMBED_SIZE)
# W1.shape -> (context_size * embed_dim, n_neurons_in_first_layer)
# b1.shape -> n_nuerons in first layer (each neuron has its own bias term)
C.shape, W1.shape, b1.shape, W2.shape, b2.shape

In [362]:
# complete training loop

for _ in range(10_000):
    
    # get mini batch
    ix = torch.randint(0, X.shape[0], size=(BATCH_SIZE,))


    # forward pass
    # embed the input contexts
    emb = C[X[ix]]

    # run the embeddings through the first layer

    h = torch.tanh((emb.view(-1, BLOCK_SIZE * EMBED_SIZE) @ W1) + b1)

    # get the logits from the output layer
    logits = (h @ W2) + b2

    # calculate the loss
    loss = F.cross_entropy(logits, Y[ix])

    # backward pass

    # zero out the gradients
    for p in parameters:
        p.grad = None



    # backprop
    loss.backward()

    # update
    for p in parameters:
        p.data = p.data - (LEARNING_RATE) * p.grad    

In [ ]:
# embed the input contexts
emb = C[X]

# run the embeddings through the first layer

h = torch.tanh((emb.view(-1, BLOCK_SIZE * EMBED_SIZE) @ W1) + b1)

# get the logits from the output layer
logits = (h @ W2) + b2

# calculate the loss
loss = F.cross_entropy(logits, Y)
loss

In [364]:
# learning rate decay
decay_factor = 10

In [365]:
# complete training loop

for _ in range(10_000):
    
    # get mini batch
    ix = torch.randint(0, X.shape[0], size=(BATCH_SIZE,))


    # forward pass
    # embed the input contexts
    emb = C[X[ix]]

    # run the embeddings through the first layer

    h = torch.tanh((emb.view(-1, BLOCK_SIZE * EMBED_SIZE) @ W1) + b1)

    # get the logits from the output layer
    logits = (h @ W2) + b2

    # calculate the loss
    loss = F.cross_entropy(logits, Y[ix])

    # backward pass

    # zero out the gradients
    for p in parameters:
        p.grad = None



    # backprop
    loss.backward()

    # update
    for p in parameters:
        p.data = p.data - (LEARNING_RATE / decay_factor) * p.grad    

In [ ]:
# embed the input contexts
emb = C[X]

# run the embeddings through the first layer

h = torch.tanh((emb.view(-1, BLOCK_SIZE * EMBED_SIZE) @ W1) + b1)

# get the logits from the output layer
logits = (h @ W2) + b2

# calculate the loss
loss = F.cross_entropy(logits, Y)
loss